# 📄 Module 4.1 — PDF Logical Splitting & Classification
**Stack:** PyMuPDF · Tesseract · Qwen2.5 (Ollama/vLLM)  
**Pipeline:** Extract → Boundary Detection → Rule-Based Classify → Qwen Verify → Mislabel Detection → Split & Export

## ⚙️ Cell 1 — Cài đặt thư viện

In [35]:
# ── INSTALL ──────────────────────────────────────────────────
!pip install pymupdf pdf2image pytesseract tqdm requests -q
!apt-get install -y tesseract-ocr tesseract-ocr-vie poppler-utils -q 2>/dev/null

import fitz, re, json, uuid, time, os, requests
import concurrent.futures
from pathlib import Path
from datetime import datetime
from PIL import Image
from tqdm.auto import tqdm
import pytesseract

print("✅ Setup xong!")

Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
tesseract-ocr-vie is already the newest version (1:4.00~git30-7274cfa-1.1).
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
✅ Setup xong!


## ⚙️ Cell 2 — Config & Ground Truth

In [37]:
from google.colab import drive
import os
import shutil

mount_point = '/content/drive'

# Check if the mount point exists and is not an actual mount, but contains files.
# If it's a regular directory with files, it's safe to clear it for a fresh mount attempt.
if os.path.exists(mount_point) and not os.path.ismount(mount_point) and os.listdir(mount_point):
  print(f"Warning: Mountpoint '{mount_point}' exists and contains files but is not a mount. Clearing it.")
  for item in os.listdir(mount_point):
    item_path = os.path.join(mount_point, item)
    try:
      if os.path.isfile(item_path) or os.path.islink(item_path):
        os.remove(item_path)
      elif os.path.isdir(item_path):
        shutil.rmtree(item_path)
    except Exception as e:
      print(f"Error clearing {item_path}: {e}")

# Mount Drive (nếu đã mount rồi lệnh này sẽ thông báo đã sẵn sàng)
drive.mount(mount_point, force_remount=True)

# Ép hệ thống làm mới (refresh) lại danh sách file trong thư mục
os.listdir(os.path.join(mount_point, 'MyDrive/VNDigitizeComprehensiveSystem_Team03/dataset_module4'))

Mounted at /content/drive


['single_doc', 'mix_doc', 'document_merged2.pdf', 'output']

In [38]:
# ── CONFIG ────────────────────────────────────────────────────
PDF_PATH   = '/content/drive/MyDrive/VNDigitizeComprehensiveSystem_Team03/dataset_module4/document_merged2.pdf'
OUTPUT_DIR = '/content/drive/MyDrive/VNDigitizeComprehensiveSystem_Team03/dataset_module4/output'

# Số văn bản pháp lý: 1-3 số / 4 số năm / ký hiệu loại
RE_SO_VAN_BAN = re.compile(r'\d{1,3}/\d{4,}/[A-ZĐQĐ][\w\-]+')

# ── Tham số hiệu năng ──
OCR_DPI      = 200      # 200 đủ cho vie-LSTM, nhanh hơn 300
OCR_WORKERS  = 6        # Luồng song song cho OCR
BLANK_THRESH = 30       # Trang < N ký tự → coi là blank

# ── Ground Truth (dùng để đánh giá / phát hiện nhãn sai) ──
GT_BOUNDARIES = {1,4,7,9,29,35,43,44,50,52,54,57,59,62,63,66,75,81,103,105,108}
GT_SEPARATORS = {65}
GT_SEGMENT_TYPES = {
    1:"Quyết định", 4:"Quyết định",  7:"Quyết định",
    9:"Bản án dân sự", 29:"Quyết định", 35:"Bản án dân sự",
    43:"Bản án hình sự", 44:"Bản án hình sự", 50:"Quyết định",
    52:"Quyết định", 54:"Quyết định", 57:"Quyết định",
    59:"Quyết định", 62:"Quyết định", 63:"Quyết định",
    66:"Bản án hình sự", 75:"Bản án dân sự", 81:"Bản án hành chính",
    103:"Quyết định", 105:"Quyết định", 108:"Quyết định",
}

LABEL_SET = {"Quyết định", "Bản án dân sự", "Bản án hình sự", "Bản án hành chính"}

# Kiểm tra file
try:
    with fitz.open(PDF_PATH) as doc:
        TOTAL_PAGES = len(doc)
    print(f"✅ Config xong | {Path(PDF_PATH).name} | {TOTAL_PAGES} trang")
except Exception as e:
    print(f"⚠️  PDF_PATH chưa đúng: {e}")
    TOTAL_PAGES = 109

✅ Config xong | document_merged2.pdf | 109 trang


## 📑 Cell 3 — Text Extraction (Hybrid: Text Layer + OCR)

In [39]:
# ── EXTRACT TEXT (hybrid) ────────────────────────────────────
def _get_text(pdf_path: str, idx: int) -> str:
    """Text layer trước; fallback OCR grayscale nếu scan."""
    with fitz.open(pdf_path) as doc:
        page = doc[idx]
        txt  = page.get_text("text").strip()
        # Đếm ký tự có nghĩa (bỏ whitespace)
        if len(re.sub(r'\s+', '', txt)) >= 20:
            return txt
        # Scan → OCR  (gray + LSTM PSM-3)
        pix = page.get_pixmap(dpi=OCR_DPI, colorspace=fitz.csGRAY)
        img = Image.frombytes("L", [pix.width, pix.height], pix.samples)
        txt = pytesseract.image_to_string(img, lang='vie',
                                          config='--oem 1 --psm 3').strip()
    return txt

def _strict_header(text: str, n: int = 7) -> tuple[str, str]:
    """Lấy n dòng đầu có nội dung, loại bỏ số trang / gạch ngang."""
    lines, result, first = text.split("\n") if text else [], [], ""
    for line in lines:
        s = line.strip()
        if not s or re.match(r'^[\d\-–—\s]{1,5}$', s):
            continue
        if not first:
            first = s
        result.append(s)
        if len(result) >= n:
            break
    return " ".join(result), first

def _extract_one(args) -> dict:
    """Worker: extract 1 trang, trả dict features."""
    pdf_path, idx = args
    text  = _get_text(pdf_path, idx)
    lines = text.split("\n") if text else []
    n     = len(lines)
    header     = " ".join(l.strip() for l in lines[:max(5, int(n*0.20))] if l.strip())
    header_up  = header.upper()
    strict_hdr, first_line = _strict_header(text)
    return {
        "page_num"      : idx + 1,
        "char_count"    : len(text),
        "is_blank"      : len(text) < BLANK_THRESH,
        "header"        : header,
        "has_quoc_hieu" : "CỘNG HÒA XÃ HỘI" in header_up,
        "has_tieu_ngu"  : "HẠNH PHÚC"        in header_up,
        "has_toa_an"    : "TÒA ÁN NHÂN DÂN"  in header_up,
        "so_vb_header"  : RE_SO_VAN_BAN.findall(header),
        "text_full"     : text,
        "strict_header" : (strict_hdr, first_line),
    }

def batch_extract(pdf_path: str) -> list[dict]:
    """Song song extract toàn bộ trang, hiển thị progress bar."""
    with fitz.open(pdf_path) as doc:
        n = len(doc)
    args = [(pdf_path, i) for i in range(n)]
    with concurrent.futures.ThreadPoolExecutor(max_workers=OCR_WORKERS) as ex:
        feats = list(tqdm(ex.map(_extract_one, args), total=n,
                          desc="Extracting pages", unit="pg"))
    return feats

print("✅ Định nghĩa Extraction xong.")

✅ Định nghĩa Extraction xong.


## 🔍 Cell 4 — Boundary Detection

In [40]:
# ── BOUNDARY DETECTION ───────────────────────────────────────
_FIRST_LINE_RE = re.compile(
    r'TÒA ÁN NHÂN DÂN|VIỆN KIỂM SÁT|CỘNG HÒA XÃ HỘI'
    r'|NHÂN DANH|QUYẾT ĐỊNH\s*$|ĐÌNH CHỈ|THÔNG BÁO'
)

def _is_doc_start(strict_hdr: str, first_line: str) -> tuple[bool, str]:
    h  = strict_hdr.upper()
    fl = first_line.upper()
    if not _FIRST_LINE_RE.search(fl):
        return False, ""

    grp_a = "CỘNG HÒA XÃ HỘI" in h or "ĐỘC LẬP" in h or "HẠNH PHÚC" in h
    grp_b = "TÒA ÁN NHÂN DÂN"  in h or "VIỆN KIỂM SÁT" in h
    grp_c = (bool(RE_SO_VAN_BAN.search(strict_hdr))
             or "BẢN ÁN SỐ"          in h
             or "NHÂN DANH"           in h
             or bool(re.search(r'QUYẾT ĐỊNH\s*$', h, re.M))
             or "ĐÌNH CHỈ VỤ ÁN"     in h
             or "THÔNG BÁO THỤ LÝ"   in h)

    n_groups = sum([grp_a, grp_b, grp_c])
    threshold = 1 if (grp_a and "CỘNG HÒA" in h) else 2
    if n_groups >= threshold:
        sigs = (["quoc_hieu"] if grp_a else []) +                (["co_quan"]   if grp_b else []) +                (["dinh_danh"] if grp_c else [])
        return True, "+".join(sigs)
    return False, ""

def detect_boundary(feat: dict, prev: dict | None) -> tuple[bool, str, float]:
    if prev is None:                 return True,  "first_page",  1.0
    if feat["is_blank"]:             return False, "blank_page",  0.0
    if prev["is_blank"]:             return True,  "after_blank", 0.95

    sh, fl = feat["strict_header"]
    is_start, sig = _is_doc_start(sh, fl)
    if is_start:
        conf = 1.0 if ("quoc_hieu" in sig and "co_quan" in sig) else 0.9
        return True, sig, conf

    prev_vb = set(prev.get("so_vb_header", []))
    curr_vb = set(feat.get("so_vb_header", []))
    if curr_vb and prev_vb and not curr_vb & prev_vb:
        return True, "new_so_vb", 0.85

    return False, "continuation", 0.0

def compute_boundaries(feats: list) -> tuple[set, set]:
    boundaries, separators = set(), set()
    for i, f in enumerate(feats):
        prev = feats[i-1] if i > 0 else None
        is_bd, _, _ = detect_boundary(f, prev)
        if is_bd:             boundaries.add(f["page_num"])
        if f["is_blank"]:     separators.add(f["page_num"])
    return boundaries, separators

def group_segments(feats: list, boundaries: set, separators: set) -> list:
    segs, cur = [], []
    for f in feats:
        pn = f["page_num"]
        if f["is_blank"] or pn in separators:
            if cur: segs.append(cur); cur = []
            continue
        if pn in boundaries and cur:
            segs.append(cur); cur = []
        cur.append(f)
    if cur: segs.append(cur)
    return segs

print("✅ Định nghĩa Boundary Detection xong.")

✅ Định nghĩa Boundary Detection xong.


## 🏷️ Cell 5 — Rule-Based Classification

In [41]:
# ── RULE-BASED CLASSIFICATION ────────────────────────────────
# Ưu tiên: Ký hiệu văn bản > Keyword scoring > Fallback

_SPECIAL_CODES = {           # QĐST-HC là Quyết định, không phải Bản án hành chính
    'QĐST-HC': "Quyết định", 'QĐPT-HC': "Quyết định",
    'QĐ-PT':   "Quyết định", 'QĐ-ST':   "Quyết định",
}
_KY_HIEU_MAP = [
    (re.compile(r'/QĐ[\-\w]*',              re.I), "Quyết định"),
    (re.compile(r'/QĐST[\-\w]*',            re.I), "Quyết định"),
    (re.compile(r'/QĐPT[\-\w]*',            re.I), "Quyết định"),
    (re.compile(r'/[\w]*HC[\-]*(ST|PT|GĐT)\b', re.I), "Bản án hành chính"),
    (re.compile(r'/[\w]*HS[\-]*(ST|PT|GĐT|SĐT)\b', re.I), "Bản án hình sự"),
    (re.compile(r'/[\w]*(DS|HNGĐ|KDTM)[\-]*(ST|PT|GĐT)?\b', re.I), "Bản án dân sự"),
]
_KEYWORDS = [
    # (keyword, label, weight)
    ("BẢN ÁN HÌNH SỰ",                "Bản án hình sự",    5),
    ("VỀ TỘI",                        "Bản án hình sự",    3),
    ("BỊ CÁO",                        "Bản án hình sự",    4),
    ("PHẠM TỘI",                      "Bản án hình sự",    3),
    ("BẢN ÁN DÂN SỰ",                 "Bản án dân sự",     5),
    ("BẢN ÁN HÔN NHÂN",               "Bản án dân sự",     5),
    ("LY HÔN",                        "Bản án dân sự",     4),
    ("TRANH CHẤP HỢP ĐỒNG",           "Bản án dân sự",     3),
    ("NGUYÊN ĐƠN",                    "Bản án dân sự",     2),
    ("BỊ ĐƠN",                        "Bản án dân sự",     2),
    ("BẢN ÁN HÀNH CHÍNH",             "Bản án hành chính", 5),
    ("KHỞI KIỆN QUYẾT ĐỊNH HÀNH CHÍNH","Bản án hành chính",5),
    ("NGƯỜI BỊ KIỆN",                 "Bản án hành chính", 3),
]

def _by_so_vb(so_vb: str) -> str | None:
    su = so_vb.upper()
    for code, lbl in _SPECIAL_CODES.items():
        if code in su: return lbl
    for pat, lbl in _KY_HIEU_MAP:
        if pat.search(so_vb): return lbl
    return None

def _by_keywords(text: str) -> tuple[str | None, float]:
    scores = dict.fromkeys(LABEL_SET, 0)
    t = text.upper()
    if re.search(r'QUYẾT ĐỊNH\s*\n', t[:500]) or        re.search(r'QUYẾT ĐỊNH\s{0,5}(V/V|SỐ|ĐÌNH CHỈ)', t[:500]):
        scores["Quyết định"] += 5
    for kw, lbl, w in _KEYWORDS:
        if kw in t[:2000]: scores[lbl] += w
    best = max(scores, key=scores.get)
    if scores[best] == 0: return None, 0.0
    vals = sorted(scores.values(), reverse=True)
    margin = vals[0] - vals[1] if len(vals) > 1 else vals[0]
    return best, min(0.95, 0.6 + margin * 0.05)

def classify_rule(pages: list[dict]) -> tuple[str, float]:
    """Rule-based: ký hiệu văn bản → keyword scoring → fallback."""
    texts = [p["text_full"].upper() for p in pages[:3]]
    for t in texts:
        for sv in RE_SO_VAN_BAN.findall(t):
            lbl = _by_so_vb(sv)
            if lbl: return lbl, 0.98
    lbl, conf = _by_keywords(" ".join(texts))
    if lbl: return lbl, conf
    if len(pages) > 1:
        t2 = pages[1]["text_full"].upper()
        for sv in RE_SO_VAN_BAN.findall(t2):
            lbl = _by_so_vb(sv)
            if lbl: return lbl, 0.90
        lbl2, conf2 = _by_keywords(t2)
        if lbl2: return lbl2, max(0.5, conf2 - 0.1)
    return "Quyết định", 0.50

print("✅ Định nghĩa Rule-Based Classifier xong.")

✅ Định nghĩa Rule-Based Classifier xong.


## 🤖 Cell 6 — Qwen2.5 Classification (LLM Verifier)

In [42]:
# ── QWEN2.5 CLASSIFICATION ───────────────────────────────────
# Yêu cầu: Qwen2.5 đang chạy qua Ollama (localhost:11434)
#   docker run -d -p 11434:11434 ollama/ollama
#   docker exec -it <id> ollama pull qwen2.5:7b

QWEN_URL     = "http://localhost:11434/api/generate"
QWEN_MODEL   = "qwen2.5:7b"
QWEN_TIMEOUT = 30          # giây

_QWEN_SYSTEM = """Bạn là chuyên gia phân loại tài liệu pháp lý Việt Nam.
Chỉ trả lời đúng 1 trong 4 nhãn sau (không giải thích):
  Quyết định
  Bản án dân sự
  Bản án hình sự
  Bản án hành chính""".strip()

def _qwen_classify(text_head: str) -> tuple[str | None, float]:
    """Gọi Qwen2.5 phân loại 800 ký tự đầu của segment."""
    prompt = f"{_QWEN_SYSTEM}\n\n---\n{text_head[:800]}\n---\nNhãn:".strip()
    try:
        resp = requests.post(QWEN_URL,
            json={"model": QWEN_MODEL, "prompt": prompt,
                  "stream": False, "options": {"temperature": 0}},
            timeout=QWEN_TIMEOUT)
        if resp.status_code != 200:
            return None, 0.0
        raw = resp.json().get("response", "").strip()
        # Khớp nhãn (partial match để chịu OCR noise)
        for lbl in LABEL_SET:
            if lbl.lower() in raw.lower():
                return lbl, 0.88
        return None, 0.0
    except Exception:
        return None, 0.0

def _qwen_available() -> bool:
    try:
        r = requests.get("http://localhost:11434", timeout=3)
        return r.status_code == 200
    except Exception:
        return False

QWEN_ON = _qwen_available()
print(f"🤖 Qwen2.5 = {'✅ online' if QWEN_ON else '⚠️  offline – chỉ dùng rule-based'}")

🤖 Qwen2.5 = ⚠️  offline – chỉ dùng rule-based


## 🚨 Cell 7 — Tự động phát hiện nhãn sai

In [43]:
# ── MISLABEL DETECTOR ────────────────────────────────────────
# So sánh 3 nguồn: rule / qwen / ground-truth → flag bất đồng

def detect_mislabels(segments: list[dict], gt: dict | None = None) -> list[dict]:
    """
    segments: list kết quả classify có các key:
        page_start, type (rule), qwen_type, confidence
    gt: {page_start: label} từ Ground Truth (nếu có)
    Trả về list các segment bị flag kèm lý do.
    """
    flagged = []
    for seg in segments:
        ps    = seg["page_start"]
        rule  = seg.get("type")
        qwen  = seg.get("qwen_type")
        conf  = seg.get("confidence", 1.0)
        gt_lbl= gt.get(ps) if gt else None

        reasons = []
        # 1. Rule vs Qwen bất đồng (cả hai đều có nhãn hợp lệ)
        if qwen and qwen in LABEL_SET and rule != qwen:
            reasons.append(f"rule={rule} ≠ qwen={qwen}")
        # 2. Confidence thấp (rule-based không tự tin)
        if conf < 0.70:
            reasons.append(f"low_conf={conf:.2f}")
        # 3. So với Ground Truth
        if gt_lbl and gt_lbl != rule:
            reasons.append(f"gt={gt_lbl} ≠ pred={rule}")

        if reasons:
            flagged.append({
                "segment"    : seg.get("segment"),
                "pages"      : f"p{seg['page_start']}-{seg['page_end']}",
                "rule_label" : rule,
                "qwen_label" : qwen,
                "gt_label"   : gt_lbl,
                "confidence" : conf,
                "reasons"    : reasons,
                # Nhãn tốt hơn: ưu tiên GT > Qwen > rule
                "suggested"  : gt_lbl or (qwen if qwen in LABEL_SET else rule),
            })
    return flagged

def print_mislabel_report(flagged: list[dict]):
    if not flagged:
        print("✅ Không phát hiện nhãn sai!")
        return
    print(f"\n🚨 Phát hiện {len(flagged)} segment nghi ngờ sai nhãn:")
    print("-" * 70)
    for f in flagged:
        print(f"  Seg {f['segment']:>3} | {f['pages']:>12} | "
              f"rule={f['rule_label']:<22} qwen={str(f['qwen_label']):<22}")
        print(f"         conf={f['confidence']:.2f} | lý do: {'; '.join(f['reasons'])}")
        print(f"         ➜ gợi ý: {f['suggested']}")
    print("-" * 70)

print("✅ Định nghĩa Mislabel Detector xong.")

✅ Định nghĩa Mislabel Detector xong.


## 🚀 Cell 8 — Full Pipeline

In [44]:
# ── FULL PIPELINE ─────────────────────────────────────────────
def full_pipeline(
    pdf_path   : str,
    output_dir : str,
    gt_boundaries   : set  | None = None,
    gt_segment_types: dict | None = None,
    use_qwen   : bool = True,
) -> dict:
    t0 = time.time()
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    # ── 1. Extract ────────────────────────────────────────────
    print("📑 [1/5] Extracting text...")
    feats = batch_extract(pdf_path)

    # ── 2. Boundary ───────────────────────────────────────────
    print("🔍 [2/5] Detecting boundaries...")
    boundaries, separators = compute_boundaries(feats)
    segs = group_segments(feats, boundaries, separators)
    print(f"   → {len(segs)} segments phát hiện")

    # ── 3. Classify (rule + Qwen) ─────────────────────────────
    print("🏷️  [3/5] Classifying segments...")
    results = []
    for i, pages in enumerate(tqdm(segs, desc="Classifying", unit="seg")):
        rule_lbl, rule_conf = classify_rule(pages)

        qwen_lbl = None
        if use_qwen and QWEN_ON:
            head = " ".join(p["text_full"] for p in pages[:2])[:1200]
            qwen_lbl, _ = _qwen_classify(head)

        # Chọn nhãn cuối: Qwen thắng nếu rule_conf < 0.80 và Qwen trả về nhãn hợp lệ
        final_lbl  = rule_lbl
        final_conf = rule_conf
        if qwen_lbl and qwen_lbl in LABEL_SET and rule_conf < 0.80:
            final_lbl  = qwen_lbl
            final_conf = 0.88

        results.append({
            "segment"    : i + 1,
            "page_start" : pages[0]["page_num"],
            "page_end"   : pages[-1]["page_num"],
            "n_pages"    : len(pages),
            "type"       : final_lbl,
            "rule_type"  : rule_lbl,
            "qwen_type"  : qwen_lbl,
            "confidence" : round(final_conf, 3),
        })

    # ── 4. Mislabel Detection ─────────────────────────────────
    print("🚨 [4/5] Detecting mislabels...")
    flagged = detect_mislabels(results, gt=gt_segment_types)
    print_mislabel_report(flagged)

    # ── 5. Split PDFs & build output JSON ─────────────────────
    print("✂️  [5/5] Splitting PDFs...")
    src    = fitz.open(pdf_path)
    doc_id = str(uuid.uuid4())[:8]
    sub_docs = []

    for r in tqdm(results, desc="Saving PDFs", unit="seg"):
        lbl_safe = r["type"].replace(" ", "_")
        fname    = f"sub_{r['segment']:03d}_{lbl_safe}_p{r['page_start']}-{r['page_end']}.pdf"
        out_pdf  = fitz.open()
        out_pdf.insert_pdf(src, from_page=r["page_start"]-1, to_page=r["page_end"]-1)
        out_pdf.save(f"{output_dir}/{fname}")
        out_pdf.close()
        sub_docs.append({
            "type": r["type"], "page_start": r["page_start"], "page_end": r["page_end"]
        })
    src.close()

    elapsed = int((time.time() - t0) * 1000)
    avg_conf = round(sum(r["confidence"] for r in results) / max(len(results), 1), 3)

    payload = {
        "document_id"       : doc_id,
        "classification"    : "Tài liệu hỗn hợp",
        "summary"           : f"Đã phân loại {len(sub_docs)} tài liệu, {TOTAL_PAGES} trang, {elapsed} ms.",
        "confidence_overall": avg_conf,
        "processing_ms"     : elapsed,
        "mislabel_flags"    : len(flagged),
        "metadata"          : [],
        "sub_documents"     : sub_docs,
    }

    json_path = f"{output_dir}/result_{doc_id}.json"
    with open(json_path, "w", encoding="utf-8") as fj:
        json.dump(payload, fj, ensure_ascii=False, indent=2)

    print(f"\n✅ Hoàn tất! | {len(sub_docs)} segs | avg_conf={avg_conf} | {elapsed} ms")
    print(f"   JSON: {json_path}")
    return payload

print("✅ Định nghĩa Full Pipeline xong.")

✅ Định nghĩa Full Pipeline xong.


## ▶️ Cell 9 — Chạy Pipeline

In [45]:
# ── CHẠY ──────────────────────────────────────────────────────
result = full_pipeline(
    pdf_path         = PDF_PATH,
    output_dir       = OUTPUT_DIR,
    gt_boundaries    = GT_BOUNDARIES,
    gt_segment_types = GT_SEGMENT_TYPES,
    use_qwen         = True,       # tắt nếu Qwen chưa chạy
)

print("\n📦 Output JSON:")
print(json.dumps(result, ensure_ascii=False, indent=2))

📑 [1/5] Extracting text...


Extracting pages:   0%|          | 0/109 [00:00<?, ?pg/s]

🔍 [2/5] Detecting boundaries...
   → 25 segments phát hiện
🏷️  [3/5] Classifying segments...


Classifying:   0%|          | 0/25 [00:00<?, ?seg/s]

🚨 [4/5] Detecting mislabels...

🚨 Phát hiện 1 segment nghi ngờ sai nhãn:
----------------------------------------------------------------------
  Seg   5 |       p29-34 | rule=Bản án hình sự         qwen=None                  
         conf=0.98 | lý do: gt=Quyết định ≠ pred=Bản án hình sự
         ➜ gợi ý: Quyết định
----------------------------------------------------------------------
✂️  [5/5] Splitting PDFs...


Saving PDFs:   0%|          | 0/25 [00:00<?, ?seg/s]


✅ Hoàn tất! | 25 segs | avg_conf=0.964 | 177237 ms
   JSON: /content/drive/MyDrive/VNDigitizeComprehensiveSystem_Team03/dataset_module4/output/result_4a9c0a68.json

📦 Output JSON:
{
  "document_id": "4a9c0a68",
  "classification": "Tài liệu hỗn hợp",
  "summary": "Đã phân loại 25 tài liệu, 109 trang, 177237 ms.",
  "confidence_overall": 0.964,
  "processing_ms": 177237,
  "mislabel_flags": 1,
  "metadata": [],
  "sub_documents": [
    {
      "type": "Quyết định",
      "page_start": 1,
      "page_end": 3
    },
    {
      "type": "Quyết định",
      "page_start": 4,
      "page_end": 6
    },
    {
      "type": "Quyết định",
      "page_start": 7,
      "page_end": 8
    },
    {
      "type": "Bản án dân sự",
      "page_start": 9,
      "page_end": 28
    },
    {
      "type": "Bản án hình sự",
      "page_start": 29,
      "page_end": 34
    },
    {
      "type": "Bản án dân sự",
      "page_start": 35,
      "page_end": 42
    },
    {
      "type": "Bản án hình sự",
      "

## 📊 Cell 10 — Đánh giá so với Ground Truth

In [46]:
# ── ĐÁNH GIÁ ACCURACY (nếu có GT) ───────────────────────────
def evaluate(results: list[dict], gt_types: dict) -> None:
    matched = total = 0
    print("\n📊 Kết quả đánh giá từng segment:")
    print(f"  {'Seg':>4} {'Pages':>12} {'Predict':>22} {'GT':>22} {'OK?':>5}")
    print("  " + "-"*68)
    for r in results:
        gt = gt_types.get(r["page_start"])
        if gt is None: continue
        ok = r["type"] == gt
        matched += ok; total += 1
        mark = "✅" if ok else "❌"
        print(f"  {r['segment']:>4} p{r['page_start']}-{r['page_end']:>3}  "
              f"{r['type']:>22}  {gt:>22}  {mark}")
    acc = matched / total * 100 if total else 0
    print(f"\n  Accuracy: {matched}/{total} = {acc:.1f}%")

# Chạy evaluate sau khi gọi full_pipeline
# (cần giữ lại biến `results` hoặc parse lại từ JSON)

# Cách chạy nhanh: gọi lại pipeline, lấy intermediate results
# Hoặc tái dựng từ sub_documents
if 'result' in dir():
    subs = result.get("sub_documents", [])
    dummy_results = [
        {"segment": i+1, "page_start": s["page_start"],
         "page_end": s["page_end"], "type": s["type"]}
        for i, s in enumerate(subs)
    ]
    evaluate(dummy_results, GT_SEGMENT_TYPES)


📊 Kết quả đánh giá từng segment:
   Seg        Pages                Predict                     GT   OK?
  --------------------------------------------------------------------
     1 p1-  3              Quyết định              Quyết định  ✅
     2 p4-  6              Quyết định              Quyết định  ✅
     3 p7-  8              Quyết định              Quyết định  ✅
     4 p9- 28           Bản án dân sự           Bản án dân sự  ✅
     5 p29- 34          Bản án hình sự              Quyết định  ❌
     6 p35- 42           Bản án dân sự           Bản án dân sự  ✅
     7 p43- 43          Bản án hình sự          Bản án hình sự  ✅
     8 p44- 49          Bản án hình sự          Bản án hình sự  ✅
     9 p50- 51              Quyết định              Quyết định  ✅
    10 p52- 53              Quyết định              Quyết định  ✅
    11 p54- 54              Quyết định              Quyết định  ✅
    14 p57- 58              Quyết định              Quyết định  ✅
    15 p59- 59              Quyết đ